# P1KS development window from Naver Finance (2015–2019 only, no login)

Calibration for the power check of the planned Korea short-sale test. **Nothing dated 2020 or later is used**: flow
requests start at 2019-12-30 and page backwards, prices are requested up to 2019-12-30, and the calibration code
stops if any later date reaches it. It prints summary statistics across stocks only.

Runtime → Run all. Downloading takes roughly 30–90 minutes; it caches each stock, so if Colab disconnects, rerun
and it resumes. Send back the final printed JSON.

Calibration code: `p1ks_dev.py`, SHA-256 `032cced20b9bc00226ca1331794d682f8aedba78845d5e37ce5ac5930e367560`.

In [ ]:
import os, subprocess, sys, hashlib
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'finance-datareader', 'lxml'], check=True)
SRC = '"""P1KS development-window calibration (2015-2019 only): summary statistics for the power check.\n\nInput: for each stock, a daily DataFrame with columns\n    close   adjusted close,\n    scale   flow scale (Naver source: trailing 60-day mean trading volume, in shares),\n    fnet    foreign net purchases, in the same units as scale (Naver: foreignerPureBuyQuant, shares),\nindexed by a DatetimeIndex. Prices may start 2014-09-01; every date must be before 2020-01-01.\n\nPer stock and window (full 2015-2019, and halves split at 2017-07-01):\n    x_t   log close minus its trailing 60-day mean, divided by the stock\'s s.d. of daily log returns (2015-2019),\n    u_t   foreign net purchase / scale at t-1, winsorized at the window\'s 0.5% and 99.5% quantiles,\n    tagged rule  u_t = alpha + r u_{t-1} + w x_{t-1} + e            (OLS),\n    impact       x_t = const + sum_{j<=5} a_j x_{t-j} + kappa u_{t-1} + e   (OLS),\n    c            kappa * w / (1 - r)   (foreign budget contribution; requires |r| < 0.99),\n    S_hat        sum of coefficients of an AR(10) in x with constant (prices only).\nOutput: cross-stock summaries only (no stock-level values).\n"""\nimport numpy as np\nimport pandas as pd\n\nCUTOFF = pd.Timestamp(\'2020-01-01\')\nSTART = pd.Timestamp(\'2015-01-01\')\nSPLIT = pd.Timestamp(\'2017-07-01\')\nAR_LAGS = 10\nIMPACT_LAGS = 5\nMIN_DAYS = 200\n\n\ndef _ols(y, X):\n    X = np.column_stack([np.ones(len(y)), X])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    return beta[1:]\n\n\ndef _lags(v, k):\n    n = len(v)\n    return np.column_stack([v[k - j:n - j] for j in range(1, k + 1)])\n\n\ndef prepare(df):\n    df = df.sort_index()\n    assert df.index.max() < CUTOFF, \'data after 2019 must not be read\'\n    lc = np.log(df[\'close\'].astype(float))\n    ret = lc.diff()\n    sd = ret[ret.index >= START].std()\n    x = (lc - lc.rolling(60).mean()) / sd\n    u = df[\'fnet\'].astype(float) / df[\'scale\'].astype(float).shift(1)\n    out = pd.DataFrame({\'x\': x, \'u\': u}).replace([np.inf, -np.inf], np.nan)\n    return out[out.index >= START].dropna()\n\n\ndef window_stats(d):\n    if len(d) < MIN_DAYS:\n        return None\n    u = d[\'u\'].to_numpy().copy()\n    lo, hi = np.quantile(u, [0.005, 0.995])\n    u = np.clip(u, lo, hi)\n    x = d[\'x\'].to_numpy()\n    if np.std(u) == 0 or np.std(x) == 0:\n        return None\n    r, w = _ols(u[1:], np.column_stack([u[:-1], x[:-1]]))\n    k = IMPACT_LAGS\n    Xi = np.column_stack([_lags(x, k), u[k - 1:-1]])\n    coef = _ols(x[k:], Xi)\n    kappa = coef[-1]\n    ar = _ols(x[AR_LAGS:], _lags(x, AR_LAGS))\n    s_hat = float(np.sum(ar))\n    c = float(kappa * w / (1 - r)) if abs(r) < 0.99 else np.nan\n    return {\'r\': float(r), \'w\': float(w), \'kappa\': float(kappa), \'c\': c, \'S\': s_hat, \'n\': int(len(d))}\n\n\ndef stock_stats(df):\n    d = prepare(df)\n    return {\'full\': window_stats(d), \'h1\': window_stats(d[d.index < SPLIT]), \'h2\': window_stats(d[d.index >= SPLIT])}\n\n\ndef _q(v):\n    v = np.asarray([a for a in v if np.isfinite(a)])\n    if len(v) == 0:\n        return None\n    qs = np.quantile(v, [0.05, 0.25, 0.5, 0.75, 0.95])\n    return {\'n\': int(len(v)), \'mean\': float(v.mean()), \'sd\': float(v.std(ddof=1)) if len(v) > 1 else None,\n            \'q05\': float(qs[0]), \'q25\': float(qs[1]), \'median\': float(qs[2]), \'q75\': float(qs[3]), \'q95\': float(qs[4])}\n\n\ndef summarize(results, groups):\n    """results: ticker -> stock_stats output; groups: ticker -> \'large\' or \'other\'."""\n    out = {\'n_stocks_input\': len(results)}\n    for g in (\'large\', \'other\', \'all\'):\n        tick = [t for t in results if g == \'all\' or groups.get(t) == g]\n        full = [results[t][\'full\'] for t in tick if results[t][\'full\']]\n        both = [(results[t][\'h1\'], results[t][\'h2\']) for t in tick\n                if results[t][\'h1\'] and results[t][\'h2\']\n                and np.isfinite(results[t][\'h1\'][\'c\']) and np.isfinite(results[t][\'h2\'][\'c\'])]\n        s = {\'n_full\': len(full), \'n_both_halves\': len(both)}\n        s[\'c_full\'] = _q([f[\'c\'] for f in full])\n        s[\'r_full\'] = _q([f[\'r\'] for f in full])\n        s[\'w_full\'] = _q([f[\'w\'] for f in full])\n        s[\'kappa_full\'] = _q([f[\'kappa\'] for f in full])\n        s[\'S_full\'] = _q([f[\'S\'] for f in full])\n        s[\'days_full\'] = _q([f[\'n\'] for f in full])\n        if len(both) > 5:\n            c1 = np.array([a[\'c\'] for a, _ in both]); c2 = np.array([b[\'c\'] for _, b in both])\n            S1 = np.array([a[\'S\'] for a, _ in both]); S2 = np.array([b[\'S\'] for _, b in both])\n            dS, dc = S2 - S1, c2 - c1\n            s[\'dS_placebo\'] = _q(dS)\n            s[\'dc_placebo\'] = _q(dc)\n            s[\'corr_dS_dc\'] = float(np.corrcoef(dS, dc)[0, 1])\n            s[\'corr_c_h1_h2\'] = float(np.corrcoef(c1, c2)[0, 1])\n            s[\'spearman_c_h1_h2\'] = float(pd.Series(c1).rank().corr(pd.Series(c2).rank()))\n            s[\'days_h1\'] = _q([a[\'n\'] for a, _ in both]); s[\'days_h2\'] = _q([b[\'n\'] for _, b in both])\n        out[g] = s\n    return out\n'
assert hashlib.sha256(SRC.encode()).hexdigest() == '032cced20b9bc00226ca1331794d682f8aedba78845d5e37ce5ac5930e367560', 'calibration code changed'
open('p1ks_dev.py', 'w').write(SRC)
print('setup ok')

In [ ]:
import ast, io, json, os, pickle, random, re, time, hashlib
import numpy as np, pandas as pd, requests
import FinanceDataReader as fdr
import p1ks_dev as P

UA = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'}
END, PSTART = '20191230', '20140901'
CUT = pd.Timestamp('2020-01-01')
CACHE = '/content/p1ks_cache'
os.makedirs(CACHE, exist_ok=True)

# ---------------------------------------------------------------- universe (listings only, no market data)
cur = pd.concat([fdr.StockListing('KOSPI'), fdr.StockListing('KOSDAQ')])
cur_codes = set(cur['Code'].astype(str))
dl = fdr.StockListing('KRX-DELISTING')
dl = dl[dl['Market'].astype(str).isin(['KOSPI', 'KOSDAQ'])]
dl = dl[(pd.to_datetime(dl['ListingDate'], errors='coerce') < pd.Timestamp('2015-01-01'))
        & (pd.to_datetime(dl['DelistingDate'], errors='coerce') >= CUT)]
codes = sorted(c for c in cur_codes | set(dl['Symbol'].astype(str)) if re.fullmatch(r'\d{5}0', c))
rng = random.Random(20261127)
universe = sorted(rng.sample(codes, min(700, len(codes))))
print('candidates', len(codes), 'sampled', len(universe))

def num(v):
    if v is None:
        return np.nan
    s = str(v).replace(',', '').replace('+', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

def trend_page(code, bizdate, size):
    r = requests.get('https://m.stock.naver.com/api/stock/%s/trend' % code,
                     params={'pageSize': size, 'bizdate': bizdate}, headers=UA, timeout=20)
    r.raise_for_status()
    return r.json()

# page size: the largest of 100 / 60 / 20 that the endpoint honours
PAGE = 20
for size in (100, 60):
    try:
        j = trend_page('005930', END, size)
        if isinstance(j, list) and len(j) >= size:
            PAGE = size
            break
    except Exception:
        pass
print('trend page size', PAGE)

def flows(code):
    rows, biz, seen = [], END, set()
    for _ in range(200):
        j = trend_page(code, biz, PAGE)
        if not isinstance(j, list) or not j:
            break
        for o in j:
            d = str(o.get('bizdate', ''))[:8]
            if d and d not in seen and d < '20200101':
                seen.add(d)
                rows.append((d, num(o.get('foreignerPureBuyQuant')), num(o.get('accumulatedTradingVolume')),
                             num(o.get('closePrice'))))
        earliest = min(str(o.get('bizdate', ''))[:8] for o in j)
        if earliest <= PSTART:
            break
        nb = (pd.Timestamp(earliest) - pd.Timedelta(days=1)).strftime('%Y%m%d')
        if nb >= biz:
            break
        biz = nb
        time.sleep(0.12)
    if not rows:
        return None
    df = pd.DataFrame(rows, columns=['date', 'fnet', 'vol', 'rawclose'])
    df['date'] = pd.to_datetime(df['date'])
    return df.set_index('date').sort_index()

PRICE_SOURCE = {}
def prices(code):
    try:
        r = requests.get('https://api.finance.naver.com/siseJson.naver',
                         params={'symbol': code, 'requestType': 1, 'startTime': PSTART, 'endTime': END, 'timeframe': 'day'},
                         headers=UA, timeout=20)
        data = ast.literal_eval(r.text.strip().replace('\n', '').replace('\t', ''))
        hdr, body = data[0], data[1:]
        ci = [i for i, h in enumerate(hdr) if '종가' in str(h)][0]
        s = pd.Series({pd.Timestamp(str(b[0])): float(b[ci]) for b in body if str(b[0]) < '20200101'})
        if len(s) > 100:
            PRICE_SOURCE['siseJson'] = PRICE_SOURCE.get('siseJson', 0) + 1
            return s.sort_index()
    except Exception:
        pass
    r = requests.get('https://fchart.stock.naver.com/sise.nhn',
                     params={'symbol': code, 'timeframe': 'day', 'count': 3000, 'requestType': 0}, headers=UA, timeout=20)
    items = [it.split('|') for it in re.findall(r'data="([^"]+)"', r.text)]
    items = [it for it in items if it[0] < '20200101']          # later rows discarded on arrival
    PRICE_SOURCE['fchart'] = PRICE_SOURCE.get('fchart', 0) + 1
    return pd.Series({pd.Timestamp(it[0]): float(it[4]) for it in items}).sort_index()

def load(code):
    f = os.path.join(CACHE, code + '.pkl')
    if os.path.exists(f):
        return pickle.load(open(f, 'rb'))
    fl = flows(code)
    if fl is None or len(fl) < 300:
        out = None
    else:
        px = prices(code)
        df = fl.join(px.rename('close'), how='left')
        df['scale'] = df['vol'].rolling(60).mean()
        df = df[df.index < CUT]
        out = df
    pickle.dump(out, open(f, 'wb'))
    return out

results, failures, split_jumps, tv = {}, {}, 0, {}
for i, code in enumerate(universe):
    try:
        df = load(code)
        if df is None or df['close'].notna().sum() < 300:
            failures[code] = 'short history'
            continue
        lr = np.log(df['close']).diff().abs()
        if (lr > np.log(1.35)).any():          # beyond the +-30% daily limit: unadjusted split; drop the stock
            split_jumps += 1
            continue
        tv[code] = float((df['rawclose'] * df['vol'])[df.index >= pd.Timestamp('2015-01-01')].mean())
        results[code] = P.stock_stats(df[['close', 'scale', 'fnet']].dropna(subset=['close']))
    except Exception as e:
        failures[code] = type(e).__name__
    if i % 50 == 0:
        print(i, 'of', len(universe), '| ok', len(results), '| failures', len(failures), flush=True)

cut = np.quantile(list(tv.values()), 0.75) if tv else np.inf
groups = {c: ('large' if tv.get(c, 0) >= cut else 'other') for c in results}
summary = P.summarize(results, groups)
summary.update({'n_sampled': len(universe), 'n_ok': len(results), 'n_failures': len(failures),
                'failure_kinds': sorted(set(failures.values()))[:10], 'n_dropped_split_jumps': split_jumps,
                'trend_page_size': PAGE, 'price_source_counts': PRICE_SOURCE,
                'code_sha256': hashlib.sha256(open('p1ks_dev.py').read().encode()).hexdigest()})
print(json.dumps(summary, indent=1, ensure_ascii=False))
print('Only cross-stock summaries were printed. Nothing dated 2020 or later was used.')